In [1]:
from kaggle_handler import handler
import pandas as pd
import numpy as np

In [2]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Stacking

    Similar to Voting Ensembles:
![image](https://miro.medium.com/v2/resize:fit:1400/0*smeJhX_yKxoqyp8R.png)

    Where Model 1, Model 2, Model 3, Model 4 (base models) are combinations of different machine learning models (like SVM, Linear Models, Decision Trees, etc.).

# How It's Different From Bagging & Boosting:
- Base models are different.
- Base models don't have weights; instead, a meta-model predicts the final output.

# Methods:
1) **Hold-Out Method (Known as Blending):**
   - Divide the training data into "n" sub-parts, where "n" = Number of Model Stacks + Meta-Model.
   - One part is used to train the Meta-Model, and the rest are used to train the Model Stacks.
    
2) **K-Fold Method (Known as Stacking):**
   - Divide the training data into N equal parts (randomly).
   - From that set, one part is used for prediction, and the rest are used to train the models in all possible combinations. (This process is repeated for all the base models in the model stack.)
   - Finally, train the base models using the entire dataset.
   - Then, combine the base models and the meta-model.

# Multi-Layer Stacking:

    Multiple layers of Model Stacks are created.

# Load Dataset

In [3]:
Assets = handler(data_set='shantanugarg274/heart-prediction-dataset-quantum', Add_more=True, Folder_Name='HeartDisease')

Directory 'Assets' already exists.
Directory 'HeartDisease' already exists.


100%|██████████| 6.83k/6.83k [00:00<00:00, 3.96MB/s]

Extracting files...
Heart Prediction Quantum Dataset.csv Moved to Assets/HeartDisease folder


In [4]:
!ls Assets/HeartDisease/

'Heart Prediction Quantum Dataset.csv'


In [5]:
df = pd.read_csv('Assets/HeartDisease/Heart Prediction Quantum Dataset.csv')
df.sample(5)

,Age,Gender,BloodPressure,Cholesterol,HeartRate,QuantumPatternFeature,HeartDisease
123,77,0,164,257,109,8.632323,0
378,54,1,90,181,119,8.809234,0
417,45,1,96,252,80,7.487654,1
499,55,0,174,249,89,10.492950,0
18,53,1,150,287,115,9.717220,0


In [6]:
X_train,X_test, y_train,y_test = train_test_split(df.drop(columns=['HeartDisease']), df['HeartDisease'],
                                                  train_size=.7)

# Blending

## Further Spliting Dataset

In [7]:
X_train_1, X_train_2, y_train_1, y_train_2 = train_test_split(X_train,y_train,train_size=.7)

## Training Base and Meta Models

In [8]:
base_M1 = SVC()
base_M2 = LogisticRegression(max_iter=500)
base_M3 = RandomForestClassifier(n_estimators=500,max_depth=4)
Meta_Model = GradientBoostingClassifier()

In [9]:
base_M1.fit(X_train_1,y_train_1)
base_M2.fit(X_train_1,y_train_1)
base_M3.fit(X_train_1,y_train_1)

M1_pred = base_M1.predict(X_train_2).reshape(X_train_2.shape[0],1)
M2_pred = base_M2.predict(X_train_2).reshape(X_train_2.shape[0],1)
M3_pred = base_M3.predict(X_train_2).reshape(X_train_2.shape[0],1)

meta_x_train = np.concat((M1_pred,M2_pred,M3_pred),axis=1)

Meta_Model.fit(meta_x_train, y_train_2)

GradientBoostingClassifier()

## Predicting & Accuracy Test

In [10]:
M1_pred = base_M1.predict(X_test).reshape(X_test.shape[0],1)
M2_pred = base_M2.predict(X_test).reshape(X_test.shape[0],1)
M3_pred = base_M3.predict(X_test).reshape(X_test.shape[0],1)

meta_x_test = np.concat((M1_pred,M2_pred,M3_pred),axis=1)

In [11]:
y_pred = Meta_Model.predict(meta_x_test)
accuracy_score(y_pred=y_pred,y_true=y_test)

0.94

# Stacking

In [12]:
def train_k_fold(X_train:pd.DataFrame,y_train:pd.DataFrame,estimators,final_estimator,k=4):
    X = X_train.reset_index(drop=True)
    y = y_train.reset_index(drop=True)
    X_set = []
    y_set = []
    models = []
    indexs = np.arange(0,k,1)
    base_pred = None
    base_true = None
    M_pred = None
    M_True = None

    # Dividing the training data into k equal parts (randomly).
    for _ in range(k):
        X_set.append(X.sample(int(X.shape[0]/k)))
        y_set.append(y.sample(int(y.shape[0]/k)))
    X_set = np.array(X_set)
    y_set = np.array(y_set)

    # Training and prediction base models for all posible combination (generation new traning set based on base-model predictions)
    for name, M in estimators:
        for i in range(k):
            M.fit(np.concat(X_set[np.delete(indexs,i)]),np.concat(y_set[np.delete(indexs,i)]))
            if i == 0:
                M_pred = np.array(M.predict(X_set[i])).reshape(X_set[i].shape[0],1)
                M_True = y_set[i]
            else:
                M_pred = np.concat((M_pred, M.predict(X_set[i]).reshape(X_set[i].shape[0],1)) )
                M_True = np.concat((M_True, y_set[i]))
                
        try:
            base_pred = np.concat((base_pred, M_pred), axis=1)
            base_true = M_True
        except:
            base_pred = M_pred
            base_true = M_True

    # Training meta model
    final_estimator.fit(base_pred, base_true)

    # Training final base models
    for name, M in estimators:
        M.fit(X_train,y_train)
        models.append(M)
    models.append(final_estimator)
    
    return models

In [13]:
estimators = [
    ('svc', SVC()),
    ('lgc', LogisticRegression(max_iter=500)),
    ('ranfor', RandomForestClassifier(n_estimators=500,max_depth=4))
             ]
models = train_k_fold(X_train, y_train,estimators=estimators,final_estimator=GradientBoostingClassifier())

In [14]:
models

[SVC(),
 LogisticRegression(max_iter=500),
 RandomForestClassifier(max_depth=4, n_estimators=500),
 GradientBoostingClassifier()]

In [15]:
svc_pred = models[0].predict(X_test).reshape(X_test.shape[0],1)
lr_pred = models[1].predict(X_test).reshape(X_test.shape[0],1)
rf_pred = models[2].predict(X_test).reshape(X_test.shape[0],1)

basemodel_data = np.concat((svc_pred,lr_pred,rf_pred), axis=1)

y_pred = models[3].predict(basemodel_data)
accuracy_score(y_pred=y_pred,y_true=y_test)

0.58

# SkLearn Model

In [16]:
estimators = [
    ('svc', SVC()),
    ('lgc', LogisticRegression(max_iter=500)),
    ('ranfor', RandomForestClassifier(n_estimators=500,max_depth=4))
             ]

In [17]:
clf = StackingClassifier(estimators=estimators,
                         final_estimator=GradientBoostingClassifier(),
                         stack_method='predict',
                         cv=10,
                         n_jobs=-1)
clf.fit(X_train, y_train)

StackingClassifier(cv=10,
                   estimators=[('svc', SVC()),
                               ('lgc', LogisticRegression(max_iter=500)),
                               ('ranfor',
                                RandomForestClassifier(max_depth=4,
                                                       n_estimators=500))],
                   final_estimator=GradientBoostingClassifier(), n_jobs=-1,
                   stack_method='predict')

In [18]:
y_pred = clf.predict(X_test)
accuracy_score(y_pred=y_pred,y_true=y_test)

0.94